In [1]:
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

load_dotenv()

# 1. 모델 설정 (Ollama 로컬 사용 시 base_url과 model_name을 지정하세요)
# Ollama 사용 시: ChatOpenAI(model="qwen2.5:1.5b", base_url="http://localhost:11434/v1", api_key="ollama")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# 2. PromptTemplate 작성
template = """
사용자가 입력한 다음 재료를 활용하여 만들 수 있는 요리를 하나 추천하고 조리법을 작성해주세요.

재료: {ingredients}

[출력 형식]
추천 요리: <요리 이름>
재료: {ingredients}
조리법:
1. <단계 1>
2. <단계 2>
3. <단계 3>
"""

prompt = PromptTemplate.from_template(template)

# 3. LCEL 파이프라인 체인 결합 (| 문법)
chain = prompt | llm | StrOutputParser()

# 4. 실행 및 결과 출력
ingredients_input = "계란, 밥, 김치"
result = chain.invoke({"ingredients": ingredients_input})

print(f"입력: \"{ingredients_input}\"")
print("출력:\n")
print(result)

입력: "계란, 밥, 김치"
출력:

추천 요리: 김치볶음밥
재료: 계란, 밥, 김치
조리법:
1. 팬에 기름을 두르고 중불로 열을 가한 후, 잘게 썬 김치를 넣고 2~3분간 볶아 김치의 맛을 살립니다.
2. 남은 밥을 팬에 넣고 김치와 잘 섞이도록 볶아줍니다. 필요에 따라 간장이나 고춧가루로 간을 맞춥니다.
3. 다른 팬에 계란을 깨서 후라이를 만들고, 볶음밥 위에 올려서 완성합니다. 원하는 경우 파나 깨를 뿌려서 장식할 수 있습니다.


In [2]:
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# 1단계 체인
prompt_step1 = ChatPromptTemplate.from_messages([
    ("system", "당신은 세계적인 여행 가이드입니다. 입력받은 지역의 가장 대표적인 관광 명소 딱 1개의 이름만 단답형으로 답변하세요."),
    ("user", "{location}")
])
chain1 = prompt_step1 | llm | StrOutputParser()

# 2단계 체인
prompt_step2 = ChatPromptTemplate.from_messages([
    ("system", "당신은 문화유산 전문 설명가입니다. 전달받은 관광 명소에 대해 역사, 특징, 방문 팁을 구분하여 상세히 설명해주세요."),
    ("user", """
명소: {attraction}

다음 양식에 맞추어 항목별로 자세히 작성하세요:
역사:
1. ...
2. ...

특징:
1. ...
2. ...

방문 팁:
1. ...
2. ...
""")
])
chain2 = prompt_step2 | llm | StrOutputParser()

# 실행
location_input = "로마"
attraction_result = chain1.invoke({"location": location_input}).strip()
print(f"입력: \"{location_input}\"")
print(f"1단계 결과: \"{attraction_result}\"\n")

details_result = chain2.invoke({"attraction": attraction_result})
print("추천 2단계 결과:")
print(f"명소: {attraction_result}")
print(details_result)

입력: "로마"
1단계 결과: "콜로세움"

추천 2단계 결과:
명소: 콜로세움
명소: 콜로세움

역사:
1. 건설 및 개장: 콜로세움은 로마 황제 베스파시아누스에 의해 70년 경에 건설이 시작되어 80년 경에 그의 아들 티투스에 의해 개장되었습니다. 로마 제국의 권력을 상징하는 이 경기장은 고대 로마의 건축 기술과 예술성을 보여줍니다.
2. 사용 용도: 초기에는 검투사들의 전투, 야생 동물 사냥, 해양 전투 등 다양한 오락을 위해 사용되었습니다. 최대 50,000명의 관중을 수용할 수 있었고, 이러한 대형 경기들은 로마 시민들에게 무료로 제공되었습니다. 콜로세움은 로마 제국의 문화와 사회적 구조를 반영하는 중요한 장소였습니다.

특징:
1. 건축 구조: 콜로세움은 원형의 구조로, 포르티코와 아치로 이루어진 3개의 층이 있습니다. 각 층은 서로 다른 스타일(도리스, 이오니아, 코린트)로 장식되어 있으며, 이는 고대 로마 건축의 발전을 보여줍니다. 또한, 콜로세움의 외관은 대리석과 콘크리트로 만들어져 있으며, 로마의 건축 기술의 정수를 담고 있습니다.
2. 역사적 유산: 콜로세움은 유네스코 세계유산으로 등록되어 있으며, 로마의 상징 중 하나로 여겨집니다. 또한, 현대의 스포츠 경기장과 오락 시설의 모델이 된 곳이기도 합니다. 오늘날에도 영화, 문학, 예술 등 다양한 분야에서 영감을 주는 장소입니다.

방문 팁:
1. 사전 예약: 콜로세움은 매년 수많은 관광객이 방문하는 인기 명소이므로, 미리 온라인으로 입장권을 예약하는 것이 좋습니다. 예약 시 시간대와 가이드 투어 옵션을 선택할 수 있어 보다 효율적인 관람이 가능합니다.
2. 최적의 관람 시간: 아침 일찍이나 늦은 오후에 방문하면 상대적으로 덜 붐비는 시간을 선택할 수 있습니다. 특히 일몰 시간에 맞춰 방문하면 아름다운 경관을 감상할 수 있는 좋은 기회를 가질 수 있습니다.


In [3]:
from dotenv import load_dotenv
from langchain_core.prompts import FewShotChatMessagePromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{news}"),
    ("ai", "키워드: {keywords}")
])

examples = [
    {
        "news": "삼성전자가 내년 초에 자체적으로 개발한 인공지능(AI) 가속기를 처음으로 출시할 예정이다. 이는 AI 반도체 시장에서 지배적인 위치를 차지하고 있는 엔비디아의 독점을 도전하고, 세계 최고의 반도체 제조업체로서의 지위를 다시 확립하려는 삼성전자의 노력으로 해석된다.",
        "keywords": "삼성전자, 인공지능, 엔비디아"
    },
    {
        "news": "세계보건기구(WHO)는 최근 새로운 건강 위기에 대응하기 위해 국제 협력의 중요성을 강조했다. 전염병 대응 역량의 강화와 글로벌 보건 시스템의 개선이 필요하다고 발표했다.",
        "keywords": "세계보건기구, 건강위기, 국제협력"
    },
    {
        "news": "한국은행 금융통화위원회가 기준금리를 현 수준인 3.50%로 동결했다. 물가 상승률이 둔화세를 보이고 있지만 가계부채 증가세와 환율 변동성을 고려해 추가 관망이 필요하다고 판단했다.",
        "keywords": "한국은행, 기준금리, 가계부채"
    }
]

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 뉴스 키워드 추출 전문가입니다. 예시와 동일하게 입력된 뉴스 기사에서 핵심 키워드 3개를 추출하여 '키워드: 단어1, 단어2, 단어3' 형식으로만 출력하세요."),
    few_shot_prompt,
    ("human", "{input}")
])

chain = final_prompt | llm | StrOutputParser()

test_news = "제미나이 2.0 플래시는 현재 구글 AI 스튜디오(Google AI Studio) 및 버텍스 AI(Vertex AI)에서 제미나이 API를 통해 개발자에게 실험 모델로 제공됩니다. 모든 개발자는 멀티모달 입력 및 텍스트 출력을 사용할 수 있으며, 텍스트 음성 변환(text-to-speech) 및 네이티브 이미지 생성은 일부 파트너들을 대상으로 제공됩니다. 내년 1월에는 더 많은 모델 사이즈와 함께 일반에 공개될 예정입니다."

result = chain.invoke({"input": test_news})

print("--- 테스트 뉴스 ---")
print(test_news)
print("\n--- 예상 출력 결과 ---")
print(result)

--- 테스트 뉴스 ---
제미나이 2.0 플래시는 현재 구글 AI 스튜디오(Google AI Studio) 및 버텍스 AI(Vertex AI)에서 제미나이 API를 통해 개발자에게 실험 모델로 제공됩니다. 모든 개발자는 멀티모달 입력 및 텍스트 출력을 사용할 수 있으며, 텍스트 음성 변환(text-to-speech) 및 네이티브 이미지 생성은 일부 파트너들을 대상으로 제공됩니다. 내년 1월에는 더 많은 모델 사이즈와 함께 일반에 공개될 예정입니다.

--- 예상 출력 결과 ---
키워드: 제미나이, 구글 AI, 멀티모달
